In [ ]:
!pip install gensim
!pip install nltk
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import nltk
import re
from sklearn.tree import plot_tree
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from statsmodels.formula.api import ols
from sklearn.metrics import mean_squared_error #Alternaitve: from sklearn.metrics import root_mean_squared_error
from gensim.models.ldamodel import LdaModel
from gensim import corpora
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# Set seed for reproducibility
np.random.seed(42)  # Set seed for NumPy
random.seed(42) # Set seed for random module

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/katharinabrennig/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/katharinabrennig/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/katharinabrennig/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


1. **Load the Data**

    Laden Sie den Datensatz von folgendem Link: https://raw.githubusercontent.com/kbrennig/MODS_WS25_26/refs/heads/main/data/yelp.csv
    Speichern Sie den Datensatz in einem pandas DataFrame mit dem Namen `reviews`. Stellen Sie sicher, dass der Datensatz die Spalten text (Textdaten für das Topic Modeling), stars (Zielvariable) sowie die Spalten `id`, `user_id`, `business_id` und `data` enthalten. Lassen Sie sich hierfür die ersten Zeilen des Datensatzes ausgeben.

2. **Split the Data**
    
    Teilen Sie die Daten in Trainings- und Testdaten auf. Erstellen Sie dafür die Inputvariable `X`, welche die Informationen aus der Spalte `text`im Datensatz `reviews`enthalten soll. Die Zielvariable `y` wird als `reviews['stars']` definiert. Führen Sie anschließend einen Train-Test-Split mit 80 % Trainingsdaten und 20 % Testdaten durch und setzen Sie dabei `random_state = 42`. Speichern Sie die resultierenden Datensätze in den Variablen `X_train`, `X_test`, `y_train` und `y_test`.

3. **Preprocess the Text**
    
    Erstellen Sie eine Funktion mit dem Namen `preprocess()`, die den Text wie folgt verarbeitet: Tokenisierung mit NLTK `word_tokenize`, Stemming der Tokens mit dem `PorterStemmer`, Entfernen englischer Stopwörter mithilfe der NLTK-Stopwortliste sowie Entfernen von Satzzeichen mittels regulärer Ausdrücke `r'[^\w\s]'`. Wenden Sie diese Funktion auf die Textspalte in den Trainings- und Testdaten an und speichern Sie die resultierenden Tokens jeweils in einer neuen Spalte mit dem Namen `tokens`.

4. **Create the Dictionary and Corpus**
    
    Erstellen Sie das Wörterbuch und die Korpora für das Topic Modeling. Generieren Sie ein Gensim-Wörterbuch mit dem Namen `dictionary` auf Basis der Trainingsdaten. Filtern Sie seltene Wörter heraus, die weniger als in 5 Reviews vorkommen. Wandeln Sie anschließend die Tokens der Trainings- und Testdaten mit `dictionary.doc2bow()` um und speichern Sie diese als `corpus_train` und `corpus_test`.

5. **Train Topic Model**
    
    Trainieren Sie ein LDA-Topic-Modell mit dem Namen `model_10` auf Basis von der Trainingsdaten. Setzen Sie die Anzahl der Topics auf 10. Verwenden Sie das im vorherigen Schritt erstellte Wörterbuch. Begrenzen Sie die Anzahl der Iterationen der Einfachheit halber auf 10. Setzen Sie den random state auf 42.

6. **Extract Topic Distributions and Store it in Data Frames**
    
    Extrahieren Sie die Topic Distribution für jedes Review. Erstellen Sie hierfür eine Funktion mit dem Namen `get_document_topic_distribution`, die einen pandas DataFrame zurückgibt, in dem jede Spalte die Wahrscheinlichkeit eines Topics darstellt. Wenden Sie diese Funktion auf die Trainings- und Testdaten an. Geben Sie als Topic Model das zuvor tainierte `model_10` an. Speichern Sie die erzeugten DataFrames unter `train_topic_distributions` für die Trainingsdaten und `test_topic_distributions` für die Testdaten.

7. **Train and Fit a Decision Tree**
    
    Trainieren Sie einen Decision Tree Regressor und speichern Sie diesen als `decision_tree` ab. Setzen Sie den random state auf 42. Führen Sie kein Hyperparameter-Tuning durch.

8. **Evaluate a Decision Tree**
    
    Evaluieren Sie den Decision Tree, indem Sie Vorhersagen für die Testdaten erzeugen und diese in `predictions_tree` speichern. Berechnen Sie anschließend den Root Mean Squared Error (RMSE) und speichern Sie diesen Wert in der Variable `rmse_tree`. Geben Sie den RMSE-Wert aus.

9. **Train and Fit a Linear Regression**

    Trainieren Sie eine Lineare Regression. Damit die Trainings- und Testdaten im richtigen Format für die Lineare Regression vorliegen, so wie wir Sie in diesem Semester kennengelernt haben, führen Sie bitte folgenden Code aus, bevor Sie die Lineare Regression trainieren:

            train_topic_distributions_ols = train_topic_distributions
            train_topic_distributions_ols["stars"] = y_train.values

            test_topic_distributions_ols = test_topic_distributions
            test_topic_distributions_ols["stars"] = y_test.values

    Trainieren Sie nun eine Lineare Regression mithilfe von statsmodels OLS mit dem Namen `linear_model`. Lassen Sie alle 10 Topics mit in die Lineare Regression einfließen. Fitten Sie das Modell und speichern Sie das gefittete Modell erneut in der Variable `linear_model`.

10. **Evaluate a Linear Regression**

    Evaluieren Sie die Lineare Regression, indem Sie Vorhersagen für die Testdaten erzeugen und diese in `predictions_linear` speichern. Berechnen Sie den RMSE und speichern Sie diesen Wert in der Variable `rmse_linear`. Geben Sie den RMSE-Wert aus.

In [ ]:
# Enter your code here!

# Load the dataset
reviews = pd.read_csv("https://raw.githubusercontent.com/kbrennig/MODS_WS24_25/refs/heads/main/data/yelp.csv")
reviews.head()

#split the data into training and test set
X = reviews.drop(columns=['id','user_id','business_id', 'date', 'stars'])
y = reviews['stars']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#text preprocessing
def preprocess(text):
    tokens = nltk.word_tokenize(text)
    stemmer = nltk.stem.PorterStemmer()
    stemmed_tokens = [stemmer.stem(token) for token in tokens]
    stopwords = nltk.corpus.stopwords.words("english")
    filtered_tokens = [token for token in stemmed_tokens if token.lower() not in stopwords]
    filtered_tokens_nopunct = [re.sub(r'[^\w\s]', '', token) for token in filtered_tokens if token]

    return filtered_tokens_nopunct

X_train['tokens'] = X_train['text'].apply(preprocess)
X_test['tokens'] = X_test['text'].apply(preprocess)

#create the dictionary and the corpus for train and test set
dictionary = corpora.Dictionary(X_train['tokens'])
dictionary.filter_extremes(no_below=5)

corpus_train = [dictionary.doc2bow(text) for text in X_train['tokens']]
corpus_test = [dictionary.doc2bow(text) for text in X_test['tokens']]

#train the LDA Topic Model
k=10
model_10 = LdaModel(corpus=corpus_train, num_topics=k, id2word = dictionary, iterations=10, random_state=42)

#extract the topic distribution
def get_document_topic_distribution(model, corpus):
    return pd.DataFrame(
        [
            [prob for _, prob in model.get_document_topics(doc, minimum_probability=0)]
            for doc in corpus
        ],
        columns=[f'Topic{i+1}' for i in range(model.num_topics)]
    )

train_topic_distributions = get_document_topic_distribution(model_10, corpus_train)
test_topic_distributions = get_document_topic_distribution(model_10, corpus_test)

#train a decision tree regressor and evaluate the decision tree
decision_tree = DecisionTreeRegressor(random_state=42).fit(train_topic_distributions, y_train)
predictions_tree = decision_tree.predict(test_topic_distributions)
rmse_tree = np.sqrt(mean_squared_error(y_test, predictions_tree)) #Alternative: rmse_tree = root_mean_squared_error(y_test, tree)
print(f"RMSE des Entscheidungsbaumes: {rmse_tree}")

# add zielvariable to triaining data and test data
train_topic_distributions_ols = train_topic_distributions
train_topic_distributions_ols["stars"] = y_train.values

test_topic_distributions_ols = test_topic_distributions
test_topic_distributions_ols["stars"] = y_test.values

#train the linear regression model and evaluate linear regression
linear_model = ols(formula="stars ~ Topic1 + Topic2 + Topic3 + Topic4 + Topic5 + Topic6 + Topic7 + Topic8 + Topic9 + Topic10", data=train_topic_distributions_ols)
linear_model = linear_model.fit()
predictions_linear = linear_model.predict(test_topic_distributions_ols)
rmse_linear = np.sqrt(mean_squared_error(y_test, predictions_linear)) #Alternative: rmse_linear = root_mean_squared_error(y_test, predictions_linear)
print(f"RMSE des linearen Regressionsmodells: {rmse_linear}")

RMSE des Entscheidungsbaumes: 1.755135322418189
RMSE des linearen Regressionsmodells: 1.280517023457326


## Ergebnisse

### Was ist der RMSE für den Decision Tree und die lineare Regression?

RMSE des Entscheidungsbaums: 1.755

RMSE des linearen Regressionsmodells: 1.280


### Interpretieren und vergleichen Sie die beiden Modelle (Decision Tree vs. Lineare Regression). Welches Modell hat besser performt? Und wieso?

Die lineare Regression hat mit einem RMSE von 1.280 besser performt als der Entscheidungsbaum. Das Modell ist besser weil der RMSE kleiner ist. Der RMSE misst die durchschnittliche Abweichung der Vorhersagen von den tatsächlichen Werten. Daher gilt: je kleiner der RMSE, desto genauer sind die Vorhersagen des Modells. Somit sagt die lineare Regressin die Sternebewertungen in diesem Fall genauer vorher als der Entscheidungsbaum.


### Was kann man machen, um die Vorhersagen zu verbessern? Nennen Sie drei Möglichkeiten.
- Variation in der Anzahl an Topics
- Weitere Modelle ausprobieren, wie z.B. einen Random Forest anstelle eines Entscheidungsbaums
- Hyperparametertuning
- mehr Daten mit einbeziehen
- Kreuzvalidierung

